# Fine-tuning Gemma 4 E4B — Rick Sanchez en français (RTX 5080 / WSL2)

**Prérequis :**
- WSL2 avec Ubuntu 22.04+
- Drivers NVIDIA 570+ (pour Blackwell / sm_120)
- CUDA 12.8+
- `python load_dataset.py` déjà exécuté pour générer le dataset FR sur HuggingFace

**Empreinte VRAM estimée (RTX 5080 / 16 GB) :**
- Modèle 4-bit : ~5.5–6 GB
- Activations + LoRA + optimizer 8-bit : ~4 GB
- Total : ~10 GB → marge confortable

In [ ]:
import os
from dotenv import load_dotenv
import wandb
wandb.login()
load_dotenv()

HF_TOKEN        = os.getenv("HUGGINGFACE_TOKEN")
HF_DATASET_NAME = os.getenv("HF_DATASET_NAME", "Aurel-test/rick-et-morty-transcripts-fr-sharegpt")
HF_MODEL_NAME   = os.getenv("HF_MODEL_NAME",   "Aurel-test/RickGemma-4-E4B-fr")

MERGED_MODEL_PATH = "./merged_model"

from huggingface_hub import login
login(token=HF_TOKEN)

## 1. Chargement du dataset français

In [ ]:
from datasets import load_dataset
from unsloth import standardize_sharegpt

dataset = load_dataset(HF_DATASET_NAME, split="train")
dataset = standardize_sharegpt(dataset)
print(f"Nombre d'exemples : {len(dataset)}")
print(dataset[0])

## 2. Chargement du modèle Gemma 4 E4B (quantifié 4-bit)

In [ ]:
from unsloth import FastModel

max_seq_length = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-e4b-it",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
    full_finetuning=False,
)

## 3. Configuration LoRA

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,     # text-only, pas besoin des couches vision
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,                    # alpha = r avec RSLoRA (échelle ≈ √r)
    lora_dropout=0.05,                # régularisation utile sur petit dataset
    bias="none",
    use_rslora=True,
    random_state=3407,
)

## 4. Application du chat template (format Gemma 4, sans thinking)


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"])

## 5. Entraînement

Batch effectif = `per_device_train_batch_size × gradient_accumulation_steps` = 1 × 8 = 8  
Gemma 4 E4B ne nécessite que ~6 GB VRAM en 4-bit, laissant une large marge sur 16 GB.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,                       # incompatible avec train_on_responses_only
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,       # batch effectif = 8
        num_train_epochs=2,                  # 2-3 max sur ~2-3k paires
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        bf16=True,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_ratio=0.05,                   # ~5% des steps, plus robuste que warmup_steps fixe
        output_dir="output",
        seed=3407,
        report_to="wandb",
        run_name="rick-gemma4-e4b-fr",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

trainer.train()

## 6. Test rapide avant de sauvegarder

In [ ]:
FastModel.for_inference(model)

SYSTEM_PROMPT_FR = """Tu incarnes Rick Sanchez, un scientifique de génie interdimensionnel.
Fais preuve d'une honnêteté sans concession, d'un esprit vif et saupoudre tes propos de jargon scientifique.
N'hésite pas à recourir à l'humour noir ou à aborder des vérités existentielles, mais propose toujours une solution (même si elle sort des sentiers battus).
Réponds UNIQUEMENT en français."""

# Format multimodal requis par le Processor de Gemma 4 E4B (qui supporte vision/audio)
messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT_FR}]},
    {"role": "user",   "content": [{"type": "text", "text": "Tu es une mauvaise personne ?"}]},
]

inputs_dict = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs_dict,
    max_new_tokens=256,
    use_cache=True,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
)

new_tokens = outputs[0][inputs_dict["input_ids"].shape[-1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

## 7. Sauvegarde du modèle 
- Sauvegarde du LoRA en Local
- Fusion 4-bit et sauvegarde du modèle fusionné dans HF

In [ ]:
# (À utiliser UNIQUEMENT si tu as redémarré ton environnement)
from unsloth import FastModel

# On recharge ton modèle fraîchement entraîné
model, tokenizer = FastModel.from_pretrained(
    model_name="output/checkpoint-288", 
    max_seq_length=2048,
    load_in_4bit=True,
)

new_repo_id = "Aurel-test/RickGemma4b_fr_bf16"

print(f"Fusion des poids et envoi du modèle pur vers {new_repo_id}...")

model.push_to_hub_merged(
    new_repo_id,
    tokenizer,
    save_method="merged_bfloat16", 
    token=HF_TOKEN,
)
print("Terminé ! 🎉")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


base_model_name = "google/gemma-4-E4B-it" 
lora_model_path = "./lora_adapter" 

# --- ÉTAPE CRUCIALE ---
# On charge le modèle de base en bfloat16, SANS load_in_4bit=True
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# On applique ton LoRA par-dessus
model = PeftModel.from_pretrained(base_model, lora_model_path)

# On fusionne le tout définitivement
print("Fusion des poids en cours...")
model = model.merge_and_unload()

# On envoie cette nouvelle version "pure" sur Hugging Face (sous un nouveau nom par exemple)
new_repo_id = "Aurel-test/RickGemma4b_fr_bf16"

print(f"Fusion des poids et envoi du modèle pur vers {new_repo_id}...")

# On utilise directement l'objet 'model' et 'tokenizer' d'Unsloth 
# que tu as utilisés pour l'entraînement juste au-dessus.
model.push_to_hub_merged(
    new_repo_id,
    tokenizer,
    save_method="merged_bfloat16", # C'est ici que va l'argument !
    token=HF_TOKEN,
)

print("Terminé ! 🎉")